# House Price Prediction - Model Training

Train machine learning models to predict Vietnamese house prices using geospatial features from Supabase.


In [28]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import pickle
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

print("✅ Imports successful")


✅ Imports successful


## Step 1: Load Data from Supabase


In [29]:
# Load data
print("Loading data from Supabase...")
df = fetch_csv_from_supabase("Raw_Features")
print(f"✓ Loaded {len(df)} records")
# Convert price from VND to billion VND
if 'price_vnd' in df.columns:
    df['price_billion_vnd'] = df['price_vnd'] / 1e9
    df['price'] = df['price_billion_vnd']
print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")


Loading data from Supabase...
📖 Fetching data from Supabase table: Raw_Features
✓ Fetched 8581 rows from Raw_Features

✓ Loaded 8581 records

Shape: (8581, 46)

Columns: ['link', 'title', 'post_day', 'street', 'old_address', 'locality', 'region', 'listing_id', 'direction', 'listing_type', 'property_type', 'legal_status', 'num_floors', 'num_bedrooms', 'road_width_m', 'width_m', 'length_m', 'price_vnd', 'area_m2', 'dining_room_bin', 'kitchen_bin', 'terrace_bin', 'car_parking_bin', 'owner_listing_bin', 'locality_square', 'locality_population_density', 'lat', 'lon', 'matched_address', 'distance_to_center_km', 'nearest_school_km', 'school_count_3km', 'nearest_hospital_km', 'hospital_count_5km', 'nearest_marketplace_km', 'marketplace_count_3km', 'nearest_supermarket_km', 'supermarket_count_3km', 'nearest_mall_km', 'mall_count_3km', 'nearest_bus_stop_km', 'bus_stop_count_1km', 'nearest_metro_km', 'metro_count_5km', 'price_billion_vnd', 'price']


In [30]:
# Inspect data
print("First 5 rows:")
df.head()

First 5 rows:


,link,title,post_day,street,old_address,locality,region,listing_id,direction,listing_type,...,nearest_supermarket_km,supermarket_count_3km,nearest_mall_km,mall_count_3km,nearest_bus_stop_km,bus_stop_count_1km,nearest_metro_km,metro_count_5km,price_billion_vnd,price
0,https://alonhadat.com.vn/ban-nha-rieng-hem-xe-...,Bán nhà riêng hẻm xe hơi 4 tầng mới đẹp lung l...,2026-05-23,đường lê quang định,"Đường Lê Quang Định, Phường 14, Quận Bình Thạn...",phường bình thạnh,hồ chí minh,18269430,unknown,can_ban,...,0.127429,61,3.067850,0,0.108269,47,2.080087,7,8.90,8.90
1,https://alonhadat.com.vn/sieu-vi-tri-toa-nha-h...,SIÊU VỊ TRÍ TÒA NHÀ HẦM 10 TẦNG LÊ THÁNH TÔN-Q...,2026-05-27,đường lê thánh tôn,"Đường Lê Thánh Tôn, Phường Bến Nghé, Quận 1, H...",phường sài gòn,hồ chí minh,17891053,unknown,can_ban,...,0.331273,97,0.690744,15,0.089534,68,0.352320,7,110.00,110.00
2,https://alonhadat.com.vn/ban-gap-nha-4-tang-ng...,"BÁN GẤP NHÀ 4 TẦNG NGANG KHỦNG 4,8M – TRẦN QUA...",2026-06-30,đường trần quang diệu,"Đường Trần Quang Diệu, Phường 14, Quận 3, Hồ C...",phường nhiêu lộc,hồ chí minh,18843030,unknown,can_ban,...,0.293004,108,0.833065,16,0.144080,55,2.531594,4,7.99,7.99
3,https://alonhadat.com.vn/4x11-nha-co-2-lau-moi...,"4x11, nhà có 2 lầu mới đường số 3, giáp Thạch LAM",2026-06-30,đường số 3,"Đường Số 3, Phường Bình Hưng Hòa A, Quận Bình ...",phường bình hưng hòa,hồ chí minh,18614772,unknown,can_ban,...,1.674035,14,1.509740,1,0.521155,15,NaN,0,4.35,4.35
4,https://alonhadat.com.vn/ban-biet-thu-khu-comp...,Bán Biệt Thự Khu Compound 284 Nguyễn Trọng Tuy...,2026-05-27,đường nguyễn trọng tuyển,"Đường Nguyễn Trọng Tuyển, Phường 10, Quận Phú ...",phường phú nhuận,hồ chí minh,16693368,unknown,can_ban,...,0.256364,53,1.945490,6,0.155312,47,3.900285,4,45.00,45.00


In [31]:
# Data info
print("\nMissing values:")
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)


Missing values:


nearest_metro_km               3673
length_m                       1096
width_m                         805
nearest_mall_km                 484
num_floors                      184
nearest_supermarket_km          129
num_bedrooms                    125
nearest_hospital_km              23
road_width_m                     21
nearest_marketplace_km           21
nearest_bus_stop_km              12
locality_population_density       7
locality_square                   7
nearest_school_km                 3
price_vnd                         1
price_billion_vnd                 1
price                             1
dtype: int64

## Step 2: Prepare Data


In [32]:
# Define features and target
FEATURE_COLS = [
    'nearest_school_km', 'school_count_3km',
    'nearest_hospital_km', 'hospital_count_5km',
    'nearest_marketplace_km', 'marketplace_count_3km',
    'nearest_supermarket_km', 'supermarket_count_3km',
    'nearest_mall_km', 'mall_count_3km',
    'nearest_bus_stop_km', 'bus_stop_count_1km',
    'nearest_metro_km', 'metro_count_5km',
    'area_m2', 'distance_to_center_km',
    'locality_square', 'locality_population_density'
]

TARGET_COL = 'price_vnd'

# Check available features
available_features = [col for col in FEATURE_COLS if col in df.columns]
missing_features = [col for col in FEATURE_COLS if col not in df.columns]

print(f"✓ Available features: {len(available_features)}/{len(FEATURE_COLS)}")
if missing_features:
    print(f"⚠️  Missing: {missing_features}")


✓ Available features: 18/18


In [33]:
# Check target column
if TARGET_COL not in df.columns:
    print(f"⚠️  Target column '{TARGET_COL}' not found")
    # Try alternative name
    if 'price' in df.columns:
        df[TARGET_COL] = df['price']
        print(f"   Using 'price' column instead")

print(f"\nTarget statistics:")
print(df[TARGET_COL].describe())


Target statistics:
count    8.580000e+03
mean     3.819710e+10
std      9.045300e+10
min      8.300000e+01
25%      6.980000e+09
50%      1.350000e+10
75%      3.500000e+10
max      2.100000e+12
Name: price_vnd, dtype: float64


In [34]:
# Prepare data
# Drop rows with missing target
df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f"After dropping NaN prices: {len(df_clean)} records")

# Select features and target
X = df_clean[available_features].copy()
y = df_clean[TARGET_COL].copy()

# Handle missing values in features
# Only fill numeric columns
X = X.select_dtypes(include=["number"]).fillna(X.select_dtypes(include=["number"]).median())
print(f"After filling missing features: {len(X)} records")

# Remove outliers (price > 100 billion is unrealistic)
mask = y < 100
X_clean = X[mask]
y_clean = y[mask]
removed = len(X) - len(X_clean)
print(f"After removing outliers (>100B): {len(X_clean)} records (removed {removed})")

X, y = X_clean, y_clean


After dropping NaN prices: 8580 records
After filling missing features: 8580 records
After removing outliers (>100B): 1 records (removed 8579)


## Step 3: Split Data


In [35]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nFeatures used: {len(available_features)}")
print(f"Features: {available_features}")

ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## Step 4: Train Models


In [ ]:
# Train Random Forest
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predict
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluate
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
rf_mae = mean_absolute_error(y_test, y_test_pred_rf)

print(f"✓ Random Forest trained")
print(f"  Train R²: {rf_train_r2:.4f}")
print(f"  Test R²: {rf_test_r2:.4f}")
print(f"  RMSE: {rf_rmse:.4f} billion VND")
print(f"  MAE: {rf_mae:.4f} billion VND")

In [ ]:
# Train Gradient Boosting
print("Training Gradient Boosting Regressor...")
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    random_state=42
)
gb_model.fit(X_train, y_train)

# Predict
y_train_pred_gb = gb_model.predict(X_train)
y_test_pred_gb = gb_model.predict(X_test)

# Evaluate
gb_train_r2 = r2_score(y_train, y_train_pred_gb)
gb_test_r2 = r2_score(y_test, y_test_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_gb))
gb_mae = mean_absolute_error(y_test, y_test_pred_gb)

print(f"✓ Gradient Boosting trained")
print(f"  Train R²: {gb_train_r2:.4f}")
print(f"  Test R²: {gb_test_r2:.4f}")
print(f"  RMSE: {gb_rmse:.4f} billion VND")
print(f"  MAE: {gb_mae:.4f} billion VND")

## Step 5: Compare Models


In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting'],
    'Train R²': [rf_train_r2, gb_train_r2],
    'Test R²': [rf_test_r2, gb_test_r2],
    'RMSE': [rf_rmse, gb_rmse],
    'MAE': [rf_mae, gb_mae]
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))

# Select best
best_model_name = 'Random Forest' if rf_test_r2 > gb_test_r2 else 'Gradient Boosting'
best_model = rf_model if rf_test_r2 > gb_test_r2 else gb_model
best_r2 = max(rf_test_r2, gb_test_r2)

print(f"\n✅ Best Model: {best_model_name} (R² = {best_r2:.4f})")

## Step 6: Feature Importance


In [ ]:
# Get feature importance from best model
if isinstance(best_model, RandomForestRegressor):
    importances = best_model.feature_importances_
elif isinstance(best_model, GradientBoostingRegressor):
    importances = best_model.feature_importances_

# Sort by importance
feature_importance = pd.DataFrame({
    'Feature': available_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'].head(10), feature_importance['Importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.tight_layout()
plt.show()

## Step 7: Prediction Analysis


In [ ]:
# Analyze predictions
y_test_pred = best_model.predict(X_test)
errors = y_test - y_test_pred

print("\nPrediction Error Statistics:")
print(f"  Mean Error: {errors.mean():.4f} billion VND")
print(f"  Std Dev: {errors.std():.4f} billion VND")
print(f"  Min Error: {errors.min():.4f} billion VND")
print(f"  Max Error: {errors.max():.4f} billion VND")
print(f"  Median Error: {np.median(errors):.4f} billion VND")

# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_test_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price (billion VND)')
axes[0].set_ylabel('Predicted Price (billion VND)')
axes[0].set_title('Actual vs Predicted Prices')
axes[0].grid(True, alpha=0.3)

# Error distribution
axes[1].hist(errors, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Prediction Error (billion VND)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prediction Errors')
axes[1].axvline(errors.mean(), color='r', linestyle='--', label=f'Mean: {errors.mean():.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Save Model


In [ ]:
# Create saved_models directory
model_dir = Path('saved_models')
model_dir.mkdir(exist_ok=True)

# Save model
model_file = best_model_name.lower().replace(' ', '_')
model_path = model_dir / f"{model_file}.joblib"
joblib.dump(best_model, model_path)
print(f"✓ Saved model: {model_path}")

# Save metadata
metadata = {
    'model_type': best_model_name,
    'features': available_features,
    'metrics': {
        'train_r2': float(rf_train_r2 if best_model_name == 'Random Forest' else gb_train_r2),
        'test_r2': float(best_r2),
        'rmse': float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
        'mae': float(mean_absolute_error(y_test, y_test_pred))
    },
    'train_size': len(X_train),
    'test_size': len(X_test)
}

meta_path = model_dir / f"{model_file}_meta.pkl"
with open(meta_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Saved metadata: {meta_path}")

print(f"\n✅ Model saved successfully!")
print(f"   Next: Use models/inference/predict.ipynb to make predictions")